In [1]:
import pdfplumber

In [3]:
with pdfplumber.open("pdfs/TalentVault_Q2_2025.pdf") as pdf:
    for page in pdf.pages:
        text = page.extract_text()          # plain text of the page
        tables = page.extract_tables()      # list of tables, each a list of rows
        for table in tables:
            for row in table:
                print(row) 

['KPI Q2 2025 Q1 2025']
['Contracted ARR $22.4M $20.8M']
['Quarterly Revenue (recognized) $5.5M $5.1M']
['Gross Margin 76% 75%']
['Gross Revenue Retention (LTM) 91% 90%']
['Net Revenue Retention (LTM) 119% 116%']
['Enterprise Accounts (>$100k ARR) 48 43']
['Total Paying Seats 6,840 6,210']
['Total Headcount 103 97']
['Monthly Net Burn ($0.68M) ($0.74M)']
['Cash Balance $17.9M $19.0M']
['Pipeline Stage # Deals Total ACV Wtd. ACV']
['Discovery / Qualified 41 $11.2M $3.4M']
['Active Evaluation 22 $8.6M $4.3M']
['Legal / Procurement 9 $4.1M $3.1M']


In [ ]:
"""
ingest.py — extract metrics from portfolio-company PDF reports and append them
to per-company JSONL files.
(design notes unchanged — see bottom for two integrity caveats worth a look)
"""

import re
import json
from pathlib import Path
from datetime import datetime, timezone

import pdfplumber

# ---------- config ----------

DATA_DIR = Path("pdfs")
OUT_DIR = Path("output/bronze")
INCLUDE_PROSE = False    # True also captures colon-delimited bullet items (low confidence)

# ---------- regexes ----------

# tightened: a decimal must be followed by digits, so a sentence-final "2025."
# no longer looks like a value token. Kills the phantom at the tokenizer.
VALUE = re.compile(r'^\(?\$?-?[\d,]+(?:\.\d+)?[MKB]?%?\)?$')
PERIOD = re.compile(r'Q[1-4]\s*\d{4}')
CID = re.compile(r'\(cid:\d+\)')
BULLET = re.compile(r'^[\s•\-\*\u2022\u25aa\u2013]+')

CURRENCY_KW = ("revenue", "cost", "fee", "spend", "depreciation",
               "amortis", "amortiz", "expense", "contribution margin", "margin per")




def detect_currency(doc_text):
    m = re.search(r'(?:reporting currency|denominated in|figures in|currency)[:\s]+([A-Z]{3})', doc_text, re.I)
    if m:
        return m.group(1).upper(), "declared"
    m = re.search(r'\bin (USD|GBP|EUR|CAD|AUD)\b', doc_text, re.I)
    if m:
        return m.group(1).upper(), "footnote"
    if "£" in doc_text: return "GBP", "symbol"
    if "€" in doc_text: return "EUR", "symbol"
    if "$" in doc_text: return "USD", "symbol"
    return None, "unknown"

# ---------- normalization ----------

def normalize_label(raw):
    if not raw:
        return ""
    s = CID.sub("", raw)                 # drop undecoded glyphs like (cid:127)
    s = BULLET.sub("", s)                # leading bullet / dash
    s = re.sub(r"\s+", " ", s).strip()
    return s.rstrip(":").strip()         # trailing colon


def is_bullet_line(line):
    s = line.strip()
    if re.match(r'\(cid:\d+\)', s):
        return True
    return s.startswith(("•", "-", "*", "\u2022", "\u25aa", "\u2013"))


def is_metric_label(label):
    """Loose gate: reject sentence-like fragments so prose can't become a metric."""
    return bool(label) and len(label) <= 60 and len(label.split()) <= 9

# ---------- parsing helpers ----------

def split_label_values(line):
    tokens = line.split()
    i = len(tokens)
    while i > 0 and VALUE.match(tokens[i - 1]):
        i -= 1
    return " ".join(tokens[:i]), tokens[i:]


def parse_value(raw):
    """'$22.4M' -> (22400000.0, pct=False, dollar=True); '76%' -> (0.76, True, False)"""
    neg = raw.startswith("(") or raw.startswith("-")
    had_pct = "%" in raw
    had_dollar = "$" in raw
    s = raw.strip("()").replace("$", "").replace(",", "").lstrip("-").rstrip("%")
    mult = 1
    if s.endswith("M"):
        mult, s = 1_000_000, s[:-1]
    elif s.endswith("K"):
        mult, s = 1_000, s[:-1]
    elif s.endswith("B"):
        mult, s = 1_000_000_000, s[:-1]
    try:
        num = float(s) * mult
    except ValueError:
        return None, had_pct, had_dollar
    if had_pct:
        num /= 100
    num = -num if neg else num
    return round(num, 6), had_pct, had_dollar


#REMOVE
def classify_unit(label, had_pct, had_dollar):
    """Unit needs the label — '1.28M' and '8.6M' are the same token."""
    l = label.lower()
    if had_pct:
        return "pct"                                   # % sign wins
    if had_dollar or any(k in l for k in CURRENCY_KW):
        return "currency"                              # revenue / cost / fee / D&A
    if "/" in label or " per " in l:
        return "ratio"                                 # Support Tickets / 1,000 Shipments
    return "count"




MONTH_Q = {1:'Q1',2:'Q1',3:'Q1', 4:'Q2',5:'Q2',6:'Q2',
           7:'Q3',8:'Q3',9:'Q3', 10:'Q4',11:'Q4',12:'Q4'}
MONTHS = {'jan':1,'feb':2,'mar':3,'apr':4,'may':5,'jun':6,
          'jul':7,'aug':8,'sep':9,'oct':10,'nov':11,'dec':12}

def parse_periods(header_line):
    # format A: explicit quarter label, e.g. "Q2 2025" -> 2025-Q2
    found = re.findall(r'(Q[1-4])\s*(\d{4})', header_line)
    if found:
        return [f"{yr}-{q}" for q, yr in found]
    # format B: "Quarter ended March 31, 2025" -> derive quarter from the month
    m = re.search(r'([A-Za-z]{3,9})\s+\d{1,2},?\s+(\d{4})', header_line)
    if m:
        mon = MONTHS.get(m.group(1)[:3].lower())
        if mon:
            return [f"{m.group(2)}-{MONTH_Q[mon]}"]
    return []



def is_header(line):
    s = line.strip()
    if s.startswith(("KPI", "Metric")) or bool(PERIOD.search(line)):
        return True
    if re.search(r'quarter ended', s, re.I):     # date-style period header
        return True
    return False


# ---------- identity ----------

def infer_company(pdf_path):
    return pdf_path.stem.split("_")[0]

# ---------- per-document extraction ----------

def extract_document(pdf_path, company, source_file, ingested_at,
                     include_prose=INCLUDE_PROSE):
    records = []
    current_periods = []
    deferred = []          # bullet lines held for the optional prose pass


    
    with pdfplumber.open(pdf_path) as pdf:
        pages_text = [p.extract_text() or "" for p in pdf.pages]   # <-- gather once

    full_text = "\n".join(pages_text)
    currency, currency_method = detect_currency(full_text)          # <-- detect once, per doc

    def emit(label, raw_val, period, confidence, snippet):
        num, had_pct, had_dollar = parse_value(raw_val)
        if num is None or not label:
            return
        records.append({
            "company": company,
            "period": period,
            "metric": label,
            "value": num,
            "unit": classify_unit(label, had_pct, had_dollar),
            "currency": currency,                 # <-- stamp on every record
            "currency_method": currency_method,   # <-- and how we found it
            "extraction_confidence": confidence,
            "source_snippet": snippet.strip(),
            "source_file": source_file,
            "ingested_at": ingested_at,
        })

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text() or ""
            for line in text.split("\n"):
                if not line.strip():
                    continue
                if is_bullet_line(line):
                    deferred.append(line)              # non-recurring items -> prose pass
                    continue

                if is_header(line):
                    periods = parse_periods(line)

                    if periods:
                        if current_periods and current_periods != periods:
                            print(
                                f"WARNING: conflicting periods: "
                                f"{current_periods} vs {periods}"
                            )

                        current_periods = periods

                    continue

                label, values = split_label_values(line)
                label = normalize_label(label)
                if not label or not values or not is_metric_label(label):
                    continue

                periods = current_periods or ["unknown"]
                aligned = len(values) == len(periods)
                for period, raw_val in zip(periods, values):
                    emit(label, raw_val,
                         period if aligned else "unknown",
                         "high" if aligned else "low",
                         line)

            # optional prose pass — colon-delimited bullets only, never "high"
            if include_prose:
                for line in deferred:
                    clean = BULLET.sub("", CID.sub("", line)).strip()
                    if ":" not in clean:
                        continue                       # e.g. rebrand-cost line has no colon
                    lab, _, tail = clean.partition(":")
                    label = normalize_label(lab)
                    if not is_metric_label(label):
                        continue
                    m = re.search(r'-?\$?[\d,]+(?:\.\d+)?[MKB]?%?', tail)
                    if m:
                        emit(label, m.group(0), "unknown", "low", clean)
            deferred.clear()

    return records

# ---------- append-only writer ----------

def append_records(company, records):
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    out_path = OUT_DIR / f"{company}.jsonl"
    with open(out_path, "a", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec) + "\n")
    return out_path

# ---------- driver ----------

def main():
    pdfs = sorted(DATA_DIR.glob("*.pdf"))
    print(f"looking in: {DATA_DIR.resolve()}")
    print(f"found {len(pdfs)} PDF(s)\n")

    for pdf_path in pdfs:
        company = infer_company(pdf_path)
        source_file = pdf_path.name
        ingested_at = datetime.now(timezone.utc).isoformat()

        records = extract_document(pdf_path, company, source_file, ingested_at)
        out_path = append_records(company, records)
        print(f"{source_file:35s} -> {out_path.name:25s} (+{len(records)} records)")


if __name__ == "__main__":
    main()

looking in: C:\Users\user\Desktop\sagard-case study\pdfs
found 24 PDF(s)

ApexFreight_Q2_2025.pdf             -> ApexFreight.jsonl         (+12 records)
CarbonTrack_Q2_2025.pdf             -> CarbonTrack.jsonl         (+24 records)
ClearPay_Q2_2025.pdf                -> ClearPay.jsonl            (+28 records)
ConstructIQ_Q2_2025.pdf             -> ConstructIQ.jsonl         (+25 records)
FleetLink_Q1_2025.pdf               -> FleetLink.jsonl           (+9 records)
FleetLink_Q4_2024.pdf               -> FleetLink.jsonl           (+11 records)
LendBridge_Q1_2025.pdf              -> LendBridge.jsonl          (+14 records)
LendBridge_Q2_2024.pdf              -> LendBridge.jsonl          (+15 records)
LendBridge_Q2_2025.pdf              -> LendBridge.jsonl          (+15 records)
LendBridge_Q3_2024.pdf              -> LendBridge.jsonl          (+15 records)
LendBridge_Q4_2024.pdf              -> LendBridge.jsonl          (+9 records)
MediSight_Q1_2025.pdf               -> MediSight.jsonl     